<a href="https://colab.research.google.com/github/Muffalo52/anima_lora-Colab-Trainer/blob/dev/anima_lora_trainer_test2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 🚀 Anima lora 고효율 학습 파이프라인 (Based on : https://github.com/sorryhyun/anima_lora)

# @markdown ### 1. 🔑 인증 설정 (선택)
HF_TOKEN = "" # @param {type:"string"}
WANDB_API_KEY = "" # @param {type:"string"}

# @markdown ### 2. 📁 프로젝트 및 데이터셋 설정
# @markdown `MyDrive/lora_training/datasets/` 경로 내에 있는 폴더명 또는 zip 파일명을 입력하세요.
USE_GOOGLE_DRIVE = True
PROJECT_NAME = "test_dataset" # @param {type:"string"}

# @markdown ### 3. ⚙️ 학습 환경 설정
MIXED_PRECISION = "fp16" # @param ["bf16", "fp16"]
TARGET_COMMIT = "1b2ecc7c7e304bb4b38d861fe78f12401112ecf0" # @param ["HEAD", "1b2ecc7c7e304bb4b38d861fe78f12401112ecf0"]
TORCH_COMPILE = True # @param {type:"boolean"}
COMPILE_INDUCTOR_MODE = "default" # @param ["default", "reduce-overhead", "max-autotune"]
# @markdown `IS_T4_GPU`를 체크하면 코랩 무료/기본 GPU(T4) 환경에 맞춰 FlashAttention을 비활성화하고 호환 모드로 안전하게 구동합니다.
IS_T4_GPU = True # @param {type:"boolean"}
GRADIENT_CHECKPOINTING = False # @param {type:"boolean"}
# @markdown 기존에 학습된 LoRA 파일(`.safetensors`)의 경로를 입력하면, 해당 가중치 위에서부터 새로운 학습을 시작합니다. (새로 학습할 경우 비워두세요)
NETWORK_WEIGHTS = "" # @param {type:"string"}
# # @markdown 기본적으로 제외되는 레이어들을 정규식으로 지정하여 강제로 학습에 포함합니다.
# # @markdown 기본적으로 학습에서 제외되는 시스템 레이어들을 어떻게 처리할지 선택합니다.
INCLUDE_MODE = "기본 (제외 유지)"

# @markdown 개별 LoRA 기능을 제어합니다.
METHOD_TYPE = "lokr" # @param ["lora", "lokr"]
LOKR_FACTOR = 8 # @param {type:"integer"}
USE_ORTHO = False # @param {type:"boolean"}
USE_ORTHO_INIT = False # @param {type:"boolean"}
USE_TIMESTEP_MASK = False # @param {type:"boolean"}
USE_REPA = False # @param {type:"boolean"}
WEIGHT_SVD = True # @param {type:"boolean"}
# @markdown 학습 해상도 풀. 띄어쓰기로 구분하세요. (허용값: 512 768 896 1024 1280 1536)
TARGET_RES = "896 1024" # @param {type:"string"}
# @markdown ### 🌟 Autoscale (해상도 스케줄링) 설정
AUTOSCALE_MODE = "none" # @param ["none", "curriculum", "random"]
AUTOSCALE_TIERS = "512 768 1024" # @param {type:"string"}
AUTOSCALE_HIGHRES_RATIO = 0.35 # @param {type:"number"}
AUTOSCALE_RAMP = "stairs" # @param ["step", "stairs"]

# @markdown ### 4. 🎛️ 기본 하이퍼파라미터 튜닝
TRAIN_BATCH_SIZE = 1 # @param {type:"integer"}
GRADIENT_ACCUMULATION_STEPS = 4 # @param {type:"integer"}
# @markdown
LEARNING_RATE = "5e-5" # @param {type:"string"}
NETWORK_DIM = 32 # @param {type:"integer"}
NETWORK_ALPHA = 32 # @param {type:"integer"}
# @markdown
MAX_TRAIN_EPOCHS = 100 # @param {type:"integer"}
SAVE_EVERY_N_EPOCHS = 2 # @param {type:"integer"}
CHECKPOINTING_EPOCHS = 9999 # @param {type:"integer"}
# @markdown
OPTIMIZER_TYPE = "AdamW" # @param ["AdamW", "Prodigy", "prodigyplus.ProdigyPlusScheduleFree"]
LR_SCHEDULER = "cosine" # @param ["constant", "cosine", "linear"]
LR_WARMUP_RATIO = 0.05 # @param {type:"slider", min:0.0, max:0.2, step:0.01}
TIMESTEP_SAMPLING = "sigmoid" #@param ["uniform", "sigmoid", "shift", "flux_shift"]
SIGMOID_SCALE = 1.2 #@param {type:"number"}
DISCRETE_FLOW_SHIFT = 1.0 # @param {type:"number"}
# @markdown
BLOCKS_TO_SWAP = 0 # @param {type:"integer"}

# @markdown ### 5. 🧠 Prodigy 옵티마이저 전용 설정
prodigy_d_coef = 1.0 # @param {type:"number"}
prodigy_d0 = 1e-6 # @param {type:"number"}
prodigy_schedulefree_c = 0 # @param {type:"number"}

# @markdown ### 6. 📝 캡션 및 텍스트 증강 설정
CAPTION_DROPOUT_RATE = 0.1 # @param {type:"number"}

# @markdown ### 7. 🎭 SAM3 마스킹 (Masked Loss) 설정
# @markdown `USE_MASKED_LOSS`를 켜면, 지정한 프롬프트에 따라 배경이나 노이즈를 학습에서 제외합니다.
USE_MASKED_LOSS = False # @param {type:"boolean"}
MASK_THRESHOLD = 0.25 # @param {type:"number"}
# @markdown 무시할 요소 (예: `text, speech bubble, watermark`). 쉼표로 구분.
MASK_IGNORE_PROMPTS = "" # @param {type:"string"}
# @markdown 집중할 대상 (예: `1girl, character`). 비워두면 무시 요소만 삭제합니다.
MASK_FOCUS_PROMPTS = "" # @param {type:"string"}

import os
import shutil
import subprocess
import re

# 경로 변수 설정
DRIVE_BASE = "/content/drive/MyDrive/lora_training"
DATASET_FOLDER = os.path.join(DRIVE_BASE, "datasets", PROJECT_NAME)
DATASET_ZIP = DATASET_FOLDER + ".zip"
OUTPUT_DIR = os.path.join(DRIVE_BASE, "output", PROJECT_NAME)
LOCAL_DATASET = "/content/anima_lora/image_dataset"

# 1. 드라이브 마운트 및 출력 폴더 생성
if USE_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. 환경 변수 설정
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
os.environ['PYTHONWARNINGS'] = "ignore"
if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN

# 3 & 4. 환경 구축 및 의존성 설치
print("\n=== [1~2/5] Environment Setup and Dependencies ===")
!apt install -y pigz aria2 -qq
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv python install 3.13

REPO_URL = "https://github.com/sorryhyun/anima_lora.git"

if not os.path.exists('/content/anima_lora'):
    !git clone {REPO_URL} /content/anima_lora
os.chdir('/content/anima_lora')

if TARGET_COMMIT != "HEAD":
    print(f"체크아웃 진행 중: {TARGET_COMMIT}")
    !git checkout {TARGET_COMMIT}

if os.path.exists('.venv/bin/python'):
    print("Valid virtual environment (.venv) found. Skipping installation.")
else:
    print("Installing environment...")
    !uv venv --python 3.13 --clear
    !uv sync

!uv pip uninstall --python .venv -y rich

# 5. 모델 가중치 다운로드
print("\n=== [3/5] Downloading Pre-trained Models ===")
downloads = [
    {
        "url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors",
        "dest": "models/vae/qwen_image_vae.safetensors"
    },
    {
        "url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors",
        "dest": "models/text_encoders/qwen_3_06b_base.safetensors"
    },
    {
        "url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors",
        "dest": "models/diffusion_models/anima-base-v1.0.safetensors"
    }
]

for item in downloads:
    dest_path = item["dest"]
    url = item["url"]

    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

    if not os.path.exists(dest_path):
        print(f"⬇️ Downloading: {os.path.basename(dest_path)}")
        aria2_cmd = [
            "aria2c", "--console-log-level=warn", "-c", "-s", "16", "-x", "16", "-k", "10M",
            "-d", os.path.dirname(dest_path), "-o", os.path.basename(dest_path)
        ]
        if HF_TOKEN.strip():
            aria2_cmd.extend(["--header", f"Authorization: Bearer {HF_TOKEN.strip()}"])
        aria2_cmd.append(url)

        try:
            subprocess.run(aria2_cmd, check=True)
        except subprocess.CalledProcessError as e:
            print(f"\n💥 Error: aria2c download failed for {os.path.basename(dest_path)}. ({e})")
            raise SystemExit("다운로드에 실패하여 프로세스를 종료합니다.")
    else:
        print(f"✅ Already exists: {os.path.basename(dest_path)}")

print("✅ Model download complete.")


# 6. 데이터셋 설정
print("\n=== [4/5] Dataset Preprocessing ===")
if USE_GOOGLE_DRIVE:
    if os.path.exists(LOCAL_DATASET):
        shutil.rmtree(LOCAL_DATASET)

    if os.path.exists(DATASET_FOLDER):
        print(f"Copying folder: {DATASET_FOLDER}")
        shutil.copytree(DATASET_FOLDER, LOCAL_DATASET)
    elif os.path.exists(DATASET_ZIP):
        print(f"Extracting archive: {DATASET_ZIP}")
        shutil.unpack_archive(DATASET_ZIP, LOCAL_DATASET)
    else:
        print("Dataset not found. Skipping preprocessing.")

#warmup ratio → warmup steps
supported_types = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
images_repeats = {}
if os.path.exists(LOCAL_DATASET):
    # 최상위 폴더 확인
    top_images = [f for f in os.listdir(LOCAL_DATASET) if os.path.isfile(os.path.join(LOCAL_DATASET, f)) and f.lower().endswith(supported_types)]
    if top_images:
        images_repeats[LOCAL_DATASET] = (len(top_images), 1)

    # 하위 폴더 확인
    for item in os.listdir(LOCAL_DATASET):
        item_path = os.path.join(LOCAL_DATASET, item)
        if os.path.isdir(item_path):
            sub_images = [f for f in os.listdir(item_path) if f.lower().endswith(supported_types)]
            if sub_images:
                folder_repeats = 1
                match = re.match(r"^(\d+)_", item)
                if match:
                    folder_repeats = int(match.group(1))
                images_repeats[item_path] = (len(sub_images), folder_repeats)

pre_steps_per_epoch = sum(img * rep for (img, rep) in images_repeats.values())
steps_per_epoch = pre_steps_per_epoch / TRAIN_BATCH_SIZE if pre_steps_per_epoch > 0 else 1
actual_total_steps = int((MAX_TRAIN_EPOCHS * steps_per_epoch) / GRADIENT_ACCUMULATION_STEPS)
lr_warmup_steps = int(actual_total_steps * LR_WARMUP_RATIO)

# ------------------------------------------------------------------------------
# 훈련 파라미터 (train_args) 생성
# ------------------------------------------------------------------------------
optimizer_args_cli = ""
max_grad_norm_cli = ""

if OPTIMIZER_TYPE == "prodigyplus.ProdigyPlusScheduleFree":
    optimizer_args = ["betas=0.95,0.99", f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}"]
    if prodigy_schedulefree_c > 0:
        optimizer_args.append(f"schedulefree_c={prodigy_schedulefree_c}")
    max_grad_norm_cli = "--max_grad_norm 0.0 "

elif OPTIMIZER_TYPE == "Prodigy":
    optimizer_args = [f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}", "betas=0.9,0.99", "weight_decay=0.01", "decouple=True", "use_bias_correction=True", "safeguard_warmup=True"]

elif OPTIMIZER_TYPE == "AdamW":
    optimizer_args = ["weight_decay=0.1", "betas=0.9,0.99", "fused=True"]


optimizer_args_cli = "--optimizer_args " + " ".join([f'"{arg}"' for arg in optimizer_args]) + " "

compile_args_cli = f"--torch_compile --compile_inductor_mode {COMPILE_INDUCTOR_MODE} " if TORCH_COMPILE else ""
t4_compat_cli = '--attn_mode "torch" ' if IS_T4_GPU else ""
grad_ckpt_cli = "--gradient_checkpointing " if GRADIENT_CHECKPOINTING else ""
unsloth_offload_checkpointing_cli = "--unsloth_offload_checkpointing " if GRADIENT_CHECKPOINTING else ""
save_precision_cli = '--save_precision "fp16" ' if IS_T4_GPU else '--save_precision "bf16" '
network_weights_cli = f'--network_weights "{NETWORK_WEIGHTS.strip()}" ' if NETWORK_WEIGHTS.strip() else ""

network_args_list = []

include_patterns_val = ""
if INCLUDE_MODE == "Modulation 포함":
    include_patterns_val = ".*_modulation.*"
elif INCLUDE_MODE == "전체 포함 (모든 레이어 학습)":
    include_patterns_val = ".*"

if include_patterns_val:
    network_args_list.append(f"include_patterns={include_patterns_val}")

network_args_list.extend([
    f"use_ortho={USE_ORTHO}",
    f"use_ortho_init={USE_ORTHO_INIT}",
    f"use_timestep_mask={USE_TIMESTEP_MASK}",
    f"use_repa={USE_REPA}",
    "min_rank=8",
    "alpha_rank_scale=1.0"
])

network_args_list.append("down_init=weight_svd" if WEIGHT_SVD else "down_init=kaiming")

print(f"✅ LoRA 상세 설정 적용: use_ortho={USE_ORTHO}, use_ortho_init={USE_ORTHO_INIT}, use_timestep_mask={USE_TIMESTEP_MASK}, use_repa={USE_REPA}")

network_args_cli = ""
if network_args_list:
    args_joined = " ".join([f"'{arg}'" for arg in network_args_list])
    network_args_cli = f"--network_args {args_joined} "

wandb_cli = ""
if WANDB_API_KEY.strip():
    wandb_cli = (
        f'--log_with wandb '
        f'--log_tracker_name "Anima_LoRA_Project" '
        f'--wandb_run_name "{PROJECT_NAME}_Run" '
        f'--wandb_api_key "{WANDB_API_KEY.strip()}" '
        f'--log_every_n_steps 1 '
    )

train_args = (
    f'--output_dir "{OUTPUT_DIR}" '
    f'--output_name "{PROJECT_NAME}" '
    f'--train_batch_size {TRAIN_BATCH_SIZE} '
    f'--network_dim {NETWORK_DIM} '
    f'--network_alpha {NETWORK_ALPHA} '
    f'--learning_rate {LEARNING_RATE} '
    f'--max_train_epochs {MAX_TRAIN_EPOCHS} '
    f'--save_every_n_epochs {SAVE_EVERY_N_EPOCHS} '
    f'--checkpointing_epochs {CHECKPOINTING_EPOCHS} '
    f'--gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} '
    f'{grad_ckpt_cli}'
    f'{unsloth_offload_checkpointing_cli}'
    f'{t4_compat_cli}'
    f'--optimizer_type "{OPTIMIZER_TYPE}" '
    f'{optimizer_args_cli}'
    f'{max_grad_norm_cli}'
    f'{network_args_cli}'
    f'{compile_args_cli}'
    f'--lr_scheduler "{LR_SCHEDULER}" '
    f'--lr_warmup_steps 0 '
    f'--lr_warmup_steps {lr_warmup_steps} '
    f'--timestep_sampling {TIMESTEP_SAMPLING} '
    f'--sigmoid_scale {SIGMOID_SCALE} '
    f'--discrete_flow_shift {DISCRETE_FLOW_SHIFT} '
    f'--blocks_to_swap {BLOCKS_TO_SWAP} '
    f'--mixed_precision "{MIXED_PRECISION}" '
    f'{save_precision_cli}'
    f'{network_weights_cli}'
    f'--caption_dropout_rate {CAPTION_DROPOUT_RATE} '
    f'--console_log_simple '
    f'--validate_every_n_epochs 9999 '
    f'--validation_split_num 0 '
    f'--sample_every_n_epochs 0 '
    f'--sample_every_n_steps 0 '
    f'--no-use_cmmd '
    f'{wandb_cli}'
    f'--seed 1557 '
    f'--autoscale_mode {AUTOSCALE_MODE} '
    f'--autoscale_highres_ratio {AUTOSCALE_HIGHRES_RATIO} '
    f'--autoscale_ramp {AUTOSCALE_RAMP} '
)

# ==============================================================================
# 메인 실행 흐름
# ==============================================================================
def start_training_pipeline():
    print("\n🛠️ 내부 패치 적용 중...")

    # os.system("git checkout HEAD library/anima/models.py > /dev/null 2>&1")

    apply_general_optimizations()
    apply_lokr_patches()
    if IS_T4_GPU:
        apply_t4_and_fp16_patches()

    print("\n📦 Downloading Danbooru Tags...")
    !uv run --no-sync python tasks.py download-danbooru-tags

    print("\n🚀 Running image bucketing & VAE latent caching...")

    if AUTOSCALE_MODE != "none":
        print(f"   * Autoscale Tiers: {AUTOSCALE_TIERS}")
        !uv run --no-sync make preprocess-resize ARGS="--autoscale_tiers {AUTOSCALE_TIERS}"
    else:
        print(f"   * Target Resolutions: {TARGET_RES}")
        !uv run --no-sync make preprocess-resize ARGS="--target_res {TARGET_RES}"

#-------------------------------------------------------------------------------
    if USE_MASKED_LOSS:
        print("\n🎭 Preparing SAM3 Model & Generating Masks...")

        # 1. 고정된 로컬 가중치 디렉토리 생성
        os.makedirs("models/sam3", exist_ok=True)

        # 2. aria2c를 이용해 미러 사이트에서 config.json과 가중치를 모두 다운로드
        files_to_download = {
            "sam3.pt": "https://huggingface.co/1038lab/sam3/resolve/main/sam3.pt",
            "config.json": "https://huggingface.co/AEmotionStudio/sam3/resolve/main/config.json"
        }

        for filename, url in files_to_download.items():
            dest = os.path.abspath(f"models/sam3/{filename}")
            if not os.path.exists(dest):
                print(f"⬇️ Downloading {filename} via aria2c...")
                aria2_cmd = [
                    "aria2c", "--console-log-level=warn", "-c", "-s", "16", "-x", "16",
                    "-d", "models/sam3", "-o", filename, url
                ]
                subprocess.run(aria2_cmd, check=True)

        # 3. Config YAML 작성
        ignore_list = [f'"{p.strip()}"' for p in MASK_IGNORE_PROMPTS.split(',') if p.strip()]
        focus_list = [f'"{p.strip()}"' for p in MASK_FOCUS_PROMPTS.split(',') if p.strip()]

        mask_yaml_content = f"""
prompts: [{ ', '.join(ignore_list) }]
focus_prompts: [{ ', '.join(focus_list) }]
threshold: {MASK_THRESHOLD}
dilate: 5
"""
        with open("sam_mask.yaml", "w", encoding="utf-8") as f:
            f.write(mask_yaml_content)

        # 4. generate_masks.py 최상단에 hf_hub_download 가로채기 코드 삽입
        target_script = "scripts/preprocess/generate_masks.py"
        if os.path.exists(target_script):
            with open(target_script, "r", encoding="utf-8") as f:
                script_code = f.read()

            # 파이썬 런타임에서 HF Hub 다운로드를 요청하면 인터넷을 끊고 로컬 경로를 반환함
            patch_code = """
# --- HF HUB OFFLINE INTERCEPT PATCH ---
import os
import huggingface_hub.file_download
_orig_hf_hub_download = huggingface_hub.file_download.hf_hub_download

def _mock_hf_hub_download(repo_id, filename, *args, **kwargs):
    if repo_id == "facebook/sam3":
        if filename == "config.json":
            return os.path.abspath("models/sam3/config.json")
        if filename.endswith(".pt"):
            return os.path.abspath("models/sam3/sam3.pt")
    return _orig_hf_hub_download(repo_id, filename, *args, **kwargs)

huggingface_hub.file_download.hf_hub_download = _mock_hf_hub_download
huggingface_hub.hf_hub_download = _mock_hf_hub_download
# --------------------------------------
"""
            if "# --- HF HUB OFFLINE INTERCEPT PATCH ---" not in script_code:
                # 스크립트의 가장 윗부분에 패치 코드를 삽입하여 가장 먼저 실행되게 함
                script_code = patch_code + "\n" + script_code
                with open(target_script, "w", encoding="utf-8") as f:
                    f.write(script_code)
                print("  ✅ Injected HF Hub Intercept Patch into generate_masks.py")

        print("🚀 Executing SAM3 generator...")
        !uv run --no-sync python scripts/preprocess/generate_masks.py \
            --config sam_mask.yaml \
            --image-dir post_image_dataset/resized \
            --mask-dir post_image_dataset/masks \
            --recursive
#-------------------------------------------------------------------------------

    !uv run --no-sync make preprocess-vae
    !uv run --no-sync make preprocess-te
    # !uv run --no-sync make preprocess-pe

    if USE_REPA:
        print("\n🚀 Caching REPA Spatial PE features...")
        !uv run --no-sync make preprocess-pe ARGS='--encoder pe_spatial'

    !uv pip install --python .venv wandb prodigy-plus-schedule-free schedulefree

    print("\n=== [5/5] Starting LoRA Model Training ===")
    if METHOD_TYPE == "lokr":
        !uv run --no-sync python train.py --method lokr {train_args}
    else:
        !uv run --no-sync make lora ARGS="{train_args}"


# ==============================================================================
# 패치 코드 정의부
# ==============================================================================
def apply_lokr_patches():
    print("Applying structural LoKR dynamic hot-patches...")
    import os

    # 1. 이전 실행으로 인해 손상된 기존 로더 코드 및 TOML 원상 복구
    os.system("git checkout networks/lora_anima/factory.py networks/lora_anima/network.py 2>/dev/null")

    # 클래스 생성 시 활용할 하이퍼파라미터 주입
    os.environ["LOKR_FACTOR"] = str(LOKR_FACTOR)

    # 2. 고효율 LoKR 모듈 생성
    lokr_code = """import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from .base import BaseLoRAModule

def get_factors(dim: int, threshold: int = 8):
    for i in range(threshold, 0, -1):
        if dim % i == 0:
            return dim // i, i
    return dim, 1

class LoKRModule(BaseLoRAModule):
    def __init__(self, lora_name, org_module, multiplier=1.0, lora_dim=4, lora_alpha=1.0, **kwargs):
        # 상위 클래스 충돌 방지 인수 제거
        kwargs.pop("down_init", None)
        kwargs.pop("up_init", None)
        kwargs.pop("channel_scale", None)

        # 핫패치: 상속 구조와 무관하게 속성 존재 보장
        self.is_frozen = False

        super().__init__(lora_name, org_module, multiplier, lora_dim, lora_alpha, **kwargs)

        if not isinstance(org_module, nn.Linear):
            raise ValueError("LoKRModule only supports nn.Linear layers.")

    def reset_parameters(self):
        nn.init.normal_(self.w1, std=1.0)
        nn.init.normal_(self.w2_down, std=1.0 / self.lora_dim)
        nn.init.zeros_(self.w2_up)

    def get_weight_delta(self):
        w2 = self.w2_up @ self.w2_down
        delta_w = torch.einsum('oi,jk->ojik', self.w1, w2)
        delta_w = delta_w.reshape(self.dim_out, self.dim_in)
        return delta_w * self.scale

    def forward(self, x):
        if self.is_frozen:
            return x
        delta_w = self.get_weight_delta()
        return F.linear(x, delta_w) * self.multiplier

    def merge_to(self, org_module):
        weight_delta = self.get_weight_delta() * self.multiplier
        org_module.weight.data.add_(weight_delta.to(org_module.weight.device))

    def fuse_weight(self):
        self.is_frozen = True
"""
    os.makedirs("networks/lora_modules", exist_ok=True)
    with open("networks/lora_modules/lokr.py", "w", encoding="utf-8") as f:
        f.write(lokr_code)

    # 3. 모듈 패키지 진입점(__init__.py)에서 LoRAModule 참조 자체를 LoKR로 스왑 (가장 안전)
    init_path = "networks/lora_modules/__init__.py"
    if os.path.exists(init_path):
        with open(init_path, "r", encoding="utf-8") as f:
            init_content = f.read()

        # 중복 등록 방지 정제
        lines = [line for line in init_content.splitlines() if "lokr" not in line and "LoRAModule = LoKRModule" not in line]
        init_content = "\n".join(lines) + "\n"

        if METHOD_TYPE == "lokr":
            init_content += "\nfrom .lokr import LoKRModule\n"
            # 가중치 수집기(Collector)가 isinstance 검사 시 부모 클래스로 추적할 수 있도록
            # 기존 오리지널 LoRAModule 클래스 명칭을 가로채 동적 믹스인 베이스 형태로 스왑합니다.
            init_content += "from .lora import LoRAModule as OrgLoRAModule\n"
            init_content += "class LoKRPatchedModule(LoKRModule, OrgLoRAModule):\n"
            init_content += "    def __init__(self, *args, **kwargs):\n"
            init_content += "        LoKRModule.__init__(self, *args, **kwargs)\n"
            init_content += "LoRAModule = LoKRPatchedModule\n"

        with open(init_path, "w", encoding="utf-8") as f:
            f.write(init_content)
        print("  ✅ lora_modules 진입점 객체 바인딩 핫패치 적용 완료")

    # 4. 빈 검증 스킵 파일 구성하여 TOML unknown key 사전 예방
    os.makedirs("configs/methods", exist_ok=True)
    with open("configs/methods/lokr.toml", "w", encoding="utf-8") as f:
        f.write("# Verification bypass\n")
    print("  ✅ configs/methods/lokr.toml 유효성 파일 선언 완료")


def apply_general_optimizations():
    print("\n[0/3] 인자 및 로깅 제어 패치 중...")

    # 1. log.py
    log_path = 'library/log.py'
    if os.path.exists(log_path):
        with open(log_path, "r", encoding="utf-8") as f: code = f.read()
        if 'raise ImportError("Force disable rich")' in code:
            print("  ⏭️ [스킵] log.py rich 비활성화 (이미 적용됨)")
        elif 'from rich.logging import RichHandler' in code:
            code = code.replace('from rich.logging import RichHandler', 'raise ImportError("Force disable rich")')
            with open(log_path, "w", encoding="utf-8") as f: f.write(code)
            print("  ✅ [성공] log.py rich 비활성화")

    # 2. base.py
    base_py_path = "library/datasets/base.py"
    if os.path.exists(base_py_path):
        with open(base_py_path, "r", encoding="utf-8") as f: code = f.read()
        old_epoch = """                logger.info(
                    "epoch is incremented. current_epoch: {}, epoch: {}".format(
                        self.current_epoch, epoch
                    )
                )
                num_epochs = epoch - self.current_epoch"""
        new_epoch = """                num_epochs = epoch - self.current_epoch"""

        old_warn = """                logger.warning(
                    "epoch is not incremented. current_epoch: {}, epoch: {}".format(
                        self.current_epoch, epoch
                    )
                )
                self.current_epoch = epoch"""
        new_warn = """                self.current_epoch = epoch"""
        if old_epoch in code:
            code = code.replace(old_epoch, new_epoch).replace(old_warn, new_warn)
            with open(base_py_path, "w", encoding="utf-8") as f: f.write(code)
            print("  ✅ [성공] base.py 에포크 로그 삭제")
        else:
            print("  ⏭️ [스킵] base.py 에포크 로그 삭제 (이미 적용됨)")

    # 3. base.toml
    base_toml_path = "configs/base.toml"
    if os.path.exists(base_toml_path):
        with open(base_toml_path, "r", encoding="utf-8") as f: toml_content = f.read()
        new_toml_content = re.sub(r'batch_size\s*=\s*\d+', f'batch_size = {TRAIN_BATCH_SIZE}', toml_content)
        if not TORCH_COMPILE:
            new_toml_content = re.sub(r'torch_compile\s*=\s*true', 'torch_compile = false', new_toml_content)
        if "repeat_by_folder_name" in new_toml_content:
            new_toml_content = re.sub(r'repeat_by_folder_name\s*=\s*false', 'repeat_by_folder_name = true', new_toml_content)
        if USE_MASKED_LOSS:
            if "mask_dir" in new_toml_content:
                new_toml_content = re.sub(r'mask_dir\s*=\s*".*"', 'mask_dir = "post_image_dataset/masks"', new_toml_content)
            else:
                new_toml_content = new_toml_content.replace('[[dataset.subsets]]', '[[dataset.subsets]]\n  mask_dir = "post_image_dataset/masks"')
        if toml_content != new_toml_content:
            with open(base_toml_path, "w", encoding="utf-8") as f: f.write(new_toml_content)
            print("  ✅ [성공] base.toml 제어")
        else:
            print("  ⏭️ [스킵] base.toml 제어 (이미 적용됨)")

    # 4. preprocess.toml
    preprocess_toml_path = "configs/preprocess.toml"
    if os.path.exists(preprocess_toml_path):
        with open(preprocess_toml_path, "r", encoding="utf-8") as f: prep_content = f.read()
        new_prep_content = re.sub(r'min_pixels\s*=\s*[\d\.]+', 'min_pixels = 0', prep_content)
        if prep_content != new_prep_content:
            with open(preprocess_toml_path, "w", encoding="utf-8") as f: f.write(new_prep_content)
            print("  ✅ [성공] preprocess.toml 제어")
        else:
            print("  ⏭️ [스킵] preprocess.toml 제어 (이미 적용됨)")

def apply_t4_and_fp16_patches():
    print("\n🚀 T4 및 FP16 최적화 패치를 시작합니다...\n")

    print("[1/3] flash-attn 제거 중...")
    os.system("uv remove flash-attn > /dev/null 2>&1 || true")
    os.system("uv pip uninstall --python .venv -y flash-attn > /dev/null 2>&1 || true")
    print("  ✅ 완료")

    print("\n[2/3] models.py FP16 NaN 방지 패치 중...")
    models_path = "library/anima/models.py"
    if os.path.exists(models_path):
        with open(models_path, "r", encoding="utf-8") as f: code = f.read()

        # 1. RMSNorm 텐서 캐스팅
        old_rmsnorm = """    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = self._norm(x.float())
        return (output * self.weight).to(x.dtype)"""
        new_rmsnorm = """    def forward(self, x: torch.Tensor) -> torch.Tensor:
        device_type = x.device.type if isinstance(x.device.type, str) and x.device.type != 'mps' else 'cpu'
        with torch.autocast(device_type=device_type, dtype=torch.float32):
            output = self._norm(x.float()).type_as(x)
            return output * self.weight"""
        if old_rmsnorm in code:
            code = code.replace(old_rmsnorm, new_rmsnorm)
            print("  ✅ [성공] RMSNorm 텐서 캐스팅")
        else:
            print("  ⏭️ [스킵] RMSNorm 텐서 캐스팅 (이미 적용됨)")

        # 2. FinalLayer 서명 및 FP32 적용
        old_final_sig = """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ):"""
        new_final_sig = """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ):"""
        old_final_body = """        if self.use_adaln_lora:
            assert adaln_lora_B_T_3D is not None
            shift_B_T_D, scale_B_T_D = (
                self.adaln_modulation(emb_B_T_D)
                + adaln_lora_B_T_3D[:, :, : 2 * self.hidden_size]
            ).chunk(2, dim=-1)
        else:
            shift_B_T_D, scale_B_T_D = self.adaln_modulation(emb_B_T_D).chunk(2, dim=-1)"""
        new_final_body = """        device_type = x_B_T_H_W_D.device.type if isinstance(x_B_T_H_W_D.device.type, str) and x_B_T_H_W_D.device.type != 'mps' else 'cpu'
        with torch.autocast(device_type=device_type, dtype=torch.float32, enabled=use_fp32):
            if self.use_adaln_lora:
                assert adaln_lora_B_T_3D is not None
                shift_B_T_D, scale_B_T_D = (
                    self.adaln_modulation(emb_B_T_D)
                    + adaln_lora_B_T_3D[:, :, : 2 * self.hidden_size]
                ).chunk(2, dim=-1)
            else:
                shift_B_T_D, scale_B_T_D = self.adaln_modulation(emb_B_T_D).chunk(2, dim=-1)"""
        if old_final_sig in code:
            code = code.replace(old_final_sig, new_final_sig).replace(old_final_body, new_final_body)
            print("  ✅ [성공] FinalLayer 서명 및 FP32 적용")
        else:
            print("  ⏭️ [스킵] FinalLayer 서명 및 FP32 적용 (이미 적용됨)")

        # 3. Block._forward AdaLN 청크 수정
        old_block_sig = """    def _forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:"""
        new_block_sig = """    def _forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ) -> torch.Tensor:"""
        old_block_body = """        if self.use_adaln_lora:
            fused_down = self.adaln_fused_down(emb_B_T_D)
            down_self, down_cross, down_mlp = fused_down.chunk(3, dim=-1)
            shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                self.adaln_up_self_attn(down_self) + adaln_lora_B_T_3D
            ).chunk(3, dim=-1)
            shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                self.adaln_up_cross_attn(down_cross) + adaln_lora_B_T_3D
            ).chunk(3, dim=-1)
            shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                self.adaln_up_mlp(down_mlp) + adaln_lora_B_T_3D
            ).chunk(3, dim=-1)
        else:
            shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                self.adaln_modulation_self_attn(emb_B_T_D).chunk(3, dim=-1)
            )
            shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                self.adaln_modulation_cross_attn(emb_B_T_D).chunk(3, dim=-1)
            )
            shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                self.adaln_modulation_mlp(emb_B_T_D).chunk(3, dim=-1)
            )"""
        new_block_body = """        if use_fp32:
            x_B_T_H_W_D = x_B_T_H_W_D.float()

        device_type = x_B_T_H_W_D.device.type if isinstance(x_B_T_H_W_D.device.type, str) and x_B_T_H_W_D.device.type != 'mps' else 'cpu'
        with torch.autocast(device_type=device_type, dtype=torch.float32, enabled=use_fp32):
            if self.use_adaln_lora:
                fused_down = self.adaln_fused_down(emb_B_T_D)
                down_self, down_cross, down_mlp = fused_down.chunk(3, dim=-1)
                shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                    self.adaln_up_self_attn(down_self) + adaln_lora_B_T_3D
                ).chunk(3, dim=-1)
                shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                    self.adaln_up_cross_attn(down_cross) + adaln_lora_B_T_3D
                ).chunk(3, dim=-1)
                shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                    self.adaln_up_mlp(down_mlp) + adaln_lora_B_T_3D
                ).chunk(3, dim=-1)
            else:
                shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                    self.adaln_modulation_self_attn(emb_B_T_D).chunk(3, dim=-1)
                )
                shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                    self.adaln_modulation_cross_attn(emb_B_T_D).chunk(3, dim=-1)
                )
                shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                    self.adaln_modulation_mlp(emb_B_T_D).chunk(3, dim=-1)
                )"""
        if old_block_sig in code:
            code = code.replace(old_block_sig, new_block_sig).replace(old_block_body, new_block_body)
            print("  `✅ [성공] Block._forward AdaLN 청크 수정")
        else:
            print("  ⏭️ [스킵] Block._forward AdaLN 청크 수정 (이미 적용됨)")

        # 4. Block.forward 오프로드 체크포인트
        old_block_fwd_sig = """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:"""
        new_block_fwd_sig = """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ) -> torch.Tensor:"""

        old_unsloth = """                return unsloth_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                )"""
        new_unsloth = """                return unsloth_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_fp32,
                )"""

        old_cp1 = """                return torch_checkpoint(
                    create_custom_forward(self._forward),
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_reentrant=False,
                )"""
        new_cp1 = """                return torch_checkpoint(
                    create_custom_forward(self._forward),
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_fp32,
                    use_reentrant=False,
                )"""

        old_cp2 = """                return torch_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_reentrant=False,
                )"""
        new_cp2 = """                return torch_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_fp32,
                    use_reentrant=False,
                )"""

        old_eager = """            return self._forward(
                x_B_T_H_W_D,
                emb_B_T_D,
                crossattn_emb,
                attn_params,
                rope_cos_sin,
                adaln_lora_B_T_3D,
            )"""
        new_eager = """            return self._forward(
                x_B_T_H_W_D,
                emb_B_T_D,
                crossattn_emb,
                attn_params,
                rope_cos_sin,
                adaln_lora_B_T_3D,
                use_fp32,
            )"""
        if old_block_fwd_sig in code:
            code = code.replace(old_block_fwd_sig, new_block_fwd_sig)
            code = code.replace(old_unsloth, new_unsloth)
            code = code.replace(old_cp1, new_cp1)
            code = code.replace(old_cp2, new_cp2)
            code = code.replace(old_eager, new_eager)
            print("  ✅ [성공] Block.forward 오프로드 체크포인트 및 서명 변경")
        else:
            print("  ⏭️ [스킵] Block.forward 오프로드 체크포인트 및 서명 변경 (이미 적용됨)")

        # 5. Anima._run_blocks 서명 및 루프
        old_run_sig = """    def _run_blocks(
        self,
        x_padded: torch.Tensor,
        t_embedding_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params,
        capture_blocks: Optional[set] = None,
        feature_sink: Optional[dict] = None,
        stop_after_block: Optional[int] = None,
        **block_kwargs,
    ) -> torch.Tensor:"""
        new_run_sig = """    def _run_blocks(
        self,
        x_padded: torch.Tensor,
        t_embedding_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params,
        capture_blocks: Optional[set] = None,
        feature_sink: Optional[dict] = None,
        stop_after_block: Optional[int] = None,
        use_fp32: bool = False,
        **block_kwargs,
    ) -> torch.Tensor:"""
        old_run_loop = """            x = block(
                x,
                t_emb_block,
                crossattn_emb,
                attn_params,
                **block_kwargs,
            )"""
        new_run_loop = """            x = block(
                x,
                t_emb_block,
                crossattn_emb,
                attn_params,
                use_fp32=use_fp32,
                **block_kwargs,
            )"""
        if old_run_sig in code:
            code = code.replace(old_run_sig, new_run_sig).replace(old_run_loop, new_run_loop)
            print("  ✅ [성공] Anima._run_blocks 서명 및 루프 인자 추가")
        else:
            print("  ⏭️ [스킵] Anima._run_blocks 서명 및 루프 인자 추가 (이미 적용됨)")

        # 6. Anima.forward_mini_train_dit 메인 로직
        old_mini_train = r"""(x_B_T_H_W_D = self\._run_blocks\(\s*\n\s*x_B_T_H_W_D,\s*\n\s*t_embedding_B_T_D,\s*\n\s*crossattn_emb,\s*\n\s*attn_params,\s*\n\s*capture_blocks=return_block_features,\s*\n\s*feature_sink=feature_sink,\s*\n\s*stop_after_block=stop_after_block,)"""
        new_mini_train = r"use_fp32 = x_B_T_H_W_D.dtype == torch.float16\n\n        \1\n            use_fp32=use_fp32,"
        if "use_fp32 = x_B_T_H_W_D.dtype == torch.float16" not in code:
            code = re.sub(old_mini_train, new_mini_train, code)
            code = re.sub(
                r"(x_B_T_H_W_O = self\.final_layer\(\s*\n\s*x_B_T_H_W_D,\s*t_emb_final,\s*adaln_lora_B_T_3D=adaln_lora_B_T_3D\s*\n\s*\))",
                r"x_B_T_H_W_O = self.final_layer(\n            x_B_T_H_W_D, t_emb_final, adaln_lora_B_T_3D=adaln_lora_B_T_3D, use_fp32=use_fp32\n        )",
                code
            )
            print("  ✅ [성공] Anima.forward_mini_train_dit 메인 로직")
        else:
            print("  ⏭️ [스킵] Anima.forward_mini_train_dit 메인 로직 (이미 적용됨)")

        # 7. _make_dynamic_seq_forward 서명 및 전달 수정
        old_dyn_sig = """    def marked_forward(
        x_B_T_H_W_D,
        emb_B_T_D,
        crossattn_emb,
        attn_params,
        rope_cos_sin=None,
        adaln_lora_B_T_3D=None,
    ):"""
        new_dyn_sig = """    def marked_forward(
        x_B_T_H_W_D,
        emb_B_T_D,
        crossattn_emb,
        attn_params,
        rope_cos_sin=None,
        adaln_lora_B_T_3D=None,
        use_fp32: bool = False,
    ):"""
        old_dyn_body = """        return compiled_inner(
            x_B_T_H_W_D,
            emb_B_T_D,
            crossattn_emb,
            attn_params,
            rope_cos_sin,
            adaln_lora_B_T_3D,
        )"""
        new_dyn_body = """        return compiled_inner(
            x_B_T_H_W_D,
            emb_B_T_D,
            crossattn_emb,
            attn_params,
            rope_cos_sin,
            adaln_lora_B_T_3D,
            use_fp32=use_fp32,
        )"""
        if old_dyn_sig in code:
            code = code.replace(old_dyn_sig, new_dyn_sig).replace(old_dyn_body, new_dyn_body)
            print("  ✅ [성공] _make_dynamic_seq_forward 서명 및 내부 전달 수정")
        else:
            print("  ⏭️ [스킵] _make_dynamic_seq_forward 서명 및 내부 전달 수정 (이미 적용되거나 없음)")

        # 파일 저장
        with open(models_path, "w", encoding="utf-8") as f:
            f.write(code)

    print("\n[3/3] _common.py 정밀도 설정 패치 중...")
    common_path = "scripts/tasks/_common.py"
    if os.path.exists(common_path):
        with open(common_path, "r", encoding="utf-8") as f: code = f.read()

        new_code = re.sub(
            r'(mixed_precision\s*=\s*)["\']bf16["\']',
            rf'\g<1>"{MIXED_PRECISION}"',
            code
        )
        if new_code != code:
            with open(common_path, "w", encoding="utf-8") as f: f.write(new_code)
            print("  ✅ 적용됨: _common.py")
        else:
            if f'"{MIXED_PRECISION}"' in code:
                print("  ⏭️ 스킵 (이미 적용됨): _common.py")
            else:
                print("  ❌ 실패: _common.py에서 'mixed_precision = \"bf16\"' 설정을 찾을 수 없음")

    print("\n🚀 패치 스크립트 실행 종료.")

start_training_pipeline()

Mounted at /content/drive

=== [1~2/5] Environment Setup and Dependencies ===
pigz is already the newest version (2.6-1).
The following additional packages will be installed:
  libaria2-0 libc-ares2
The following NEW packages will be installed:
  aria2 libaria2-0 libc-ares2
0 upgraded, 3 newly installed, 0 to remove and 53 not upgraded.
Need to get 1,513 kB of archives.
After this operation, 5,441 kB of additional disk space will be used.
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting

In [ ]:
# @title ✔️ SAM3 마스크 테스트
# @markdown SAM3 마스킹 단계까지 완료 후 이 셀을 실행하여 마스크 구역을 확인합니다.
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# 검수할 디렉토리 경로 설정
RESIZED_DIR = "post_image_dataset/resized"
MASK_DIR = "post_image_dataset/masks"

# 최대 몇 장까지 출력할지 설정 (0으로 설정 시 제한 없이 모든 이미지 출력)
MAX_DISPLAY_COUNT = 50

# 하위 폴더 내부의 모든 이미지 탐색
image_paths = glob.glob(os.path.join(RESIZED_DIR, "**", "*.*"), recursive=True)
image_paths = [p for p in image_paths if p.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
image_paths.sort()

if not image_paths:
    print(f"❌ {RESIZED_DIR} 폴더 또는 그 하위 폴더 안에 이미지가 존재하지 않습니다.")
elif not os.path.exists(MASK_DIR) or not os.listdir(MASK_DIR):
    print(f"❌ {MASK_DIR} 폴더 안에 마스크 파일이 존재하지 않습니다. 먼저 마스크를 생성해 주세요.")
else:
    display_count = 0

    for img_p in image_paths:
        if MAX_DISPLAY_COUNT > 0 and display_count >= MAX_DISPLAY_COUNT:
            print(f"\n⚠️ 안전을 위해 최대 {MAX_DISPLAY_COUNT}장까지만 출력했습니다. 더 보려면 MAX_DISPLAY_COUNT 값을 늘리세요.")
            break

        rel_path = os.path.relpath(img_p, RESIZED_DIR)
        rel_stem, _ = os.path.splitext(rel_path)

        # 하위 폴더 경로까지 그대로 유지한 채 마스크 파일명 생성
        mask_p = os.path.join(MASK_DIR, f"{rel_stem}_mask.png")
        base_name = os.path.basename(img_p)

        if not os.path.exists(mask_p):
            folder_hint = os.path.dirname(rel_path)
            print(f"⚠️ [{folder_hint}] {base_name}에 해당하는 마스크 파일이 없어 건너뜁니다.")
            continue

        # 이미지 및 마스크 로드
        img = Image.open(img_p).convert("RGB")
        mask = Image.open(mask_p).convert("L")

        # 붉은색 오버레이 레이어 생성
        mask_np = np.array(mask)
        overlay_np = np.zeros((mask_np.shape[0], mask_np.shape[1], 4), dtype=np.uint8)

        # 무시 영역(0)을 반투명한 빨간색으로 채움
        overlay_np[mask_np == 0] = [255, 0, 0, 120]
        overlay_img = Image.fromarray(overlay_np, "RGBA")

        # 원본 이미지 위에 빨간 필터 얹기
        img_rgba = img.convert("RGBA")
        visualized_img = Image.alpha_composite(img_rgba, overlay_img)

        # ----------------------------------------------------
        # 화면 출력
        # ----------------------------------------------------
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(img)
        # 제목에 파일이 속한 하위 폴더 표시
        axes[0].set_title(f"Original ({os.path.dirname(rel_path)} / {base_name})", fontsize=10)
        axes[0].axis("off")

        axes[1].imshow(mask, cmap="gray")
        axes[1].set_title("SAM3 Binary Mask\n(White=Train, Black=Ignore)", fontsize=11)
        axes[1].axis("off")

        axes[2].imshow(visualized_img)
        axes[2].set_title("Mask Overlay Result\n(Red Area = Loss Ignored)", fontsize=11, color="red")
        axes[2].axis("off")

        plt.tight_layout()
        plt.show()

        display_count += 1